In [3]:
!pip uninstall -y torchvision torchmetrics pytorch-lightning lightning
!pip install --upgrade "torch>=2.0.0" "torchvision>=0.15.0" "lightning>=2.0.0" "torchmetrics>=1.0.0" "transformers<4.40.0"
!pip install autogluon

  Using cached lightning-2.5.6-py3-none-any.whl.metadata (42 kB)
  Using cached transformers-4.57.6-py3-none-any.whl.metadata (43 kB)
  Using cached torchvision-0.24.1-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (5.9 kB)
  Using cached torchmetrics-1.7.4-py3-none-any.whl.metadata (21 kB)
  Using cached triton-3.5.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (1.7 kB)
Using cached lightning-2.5.6-py3-none-any.whl (827 kB)
Using cached torch-2.9.1-cp312-cp312-manylinux_2_28_x86_64.whl (899.7 MB)
Using cached triton-3.5.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (170.5 MB)
Using cached torchmetrics-1.7.4-py3-none-any.whl (963 kB)
Using cached torchvision-0.24.1-cp312-cp312-manylinux_2_28_x86_64.whl (8.0 MB)
Using cached transformers-4.57.6-py3-none-any.whl (12.0 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 912.0 kB/s eta 0:00:00
  Attempting uninstall: triton
    Found existing installation: triton 3.7.0
    Uninstalling triton-

In [1]:
import pandas as pd
import numpy as np
from autogluon.tabular import TabularDataset, TabularPredictor

# =========================================================
# 1. YARDIMCI FONKSİYONLAR VE FEATURE ENGINEERING
# =========================================================
def safe_div(a, b):
    return a / np.where(b == 0, 1e-6, b)

def apply_feature_engineering(df):
    df = df.copy()

    # --- 1. Time / Career Timing ---
    if "application_year" in df.columns and "graduation_year" in df.columns:
        app_yr = pd.to_numeric(df["application_year"], errors='coerce')
        grad_yr = pd.to_numeric(df["graduation_year"], errors='coerce')
        df["years_since_graduation"] = app_yr - grad_yr
        df["is_recent_grad"] = (df["years_since_graduation"] <= 2).astype("int8")
        df["is_negative_grad_gap"] = (df["years_since_graduation"] < 0).astype("int8")

    if "age" in df.columns and "years_since_graduation" in df.columns:
        df["age_minus_years_since_grad"] = df["age"] - df["years_since_graduation"]

    # --- 2. Academic Block & Consistency ---
    academic_cols = [c for c in ["cgpa", "english_exam_score", "attendance_rate"] if c in df.columns]
    if academic_cols:
        df["academic_mean"] = df[academic_cols].mean(axis=1)
        df["academic_std"] = df[academic_cols].std(axis=1).fillna(0)
        df["academic_min"] = df[academic_cols].min(axis=1)
        df["academic_max"] = df[academic_cols].max(axis=1)

    if "cgpa" in df.columns and "attendance_rate" in df.columns:
        df['academic_consistency_score'] = (df['cgpa'] * 25) * (df['attendance_rate'] / 100)

    if "failed_courses_count" in df.columns:
        df["failed_course_flag"] = (df["failed_courses_count"] > 0).astype("int8")
        df["log_failed_courses_count"] = np.log1p(df["failed_courses_count"].clip(lower=0))

    # --- 3. Hard Skills & Tech Versatility ---
    hard_skill_cols = [c for c in [
        "coding_score", "problem_solving_score", "data_structures_score", "sql_score",
        "machine_learning_score", "backend_score", "frontend_score", "cloud_score", "devops_score"
    ] if c in df.columns]

    if hard_skill_cols:
        df["hard_skill_mean"] = df[hard_skill_cols].mean(axis=1)
        df["hard_skill_std"] = df[hard_skill_cols].std(axis=1).fillna(0)
        df["hard_skill_min"] = df[hard_skill_cols].min(axis=1)
        df["hard_skill_max"] = df[hard_skill_cols].max(axis=1)
        df["hard_skill_balance"] = df["hard_skill_max"] - df["hard_skill_min"]
        df['tech_versatility'] = (df[hard_skill_cols] > 80).sum(axis=1)

    if all(c in df.columns for c in ["coding_score", "problem_solving_score", "data_structures_score"]):
        df['core_tech_score'] = df[['coding_score', 'problem_solving_score', 'data_structures_score']].mean(axis=1)

    if all(c in df.columns for c in ["backend_score", "frontend_score"]):
        df["frontend_backend_gap"] = df["backend_score"] - df["frontend_score"]

    if all(c in df.columns for c in ["machine_learning_score", "backend_score", "frontend_score"]):
        df["ml_vs_web_gap"] = df["machine_learning_score"] - df[["backend_score", "frontend_score"]].mean(axis=1)

    if all(c in df.columns for c in ["cloud_score", "devops_score", "backend_score", "frontend_score"]):
        df["ops_vs_app_gap"] = df[["cloud_score", "devops_score"]].mean(axis=1) - df[["backend_score", "frontend_score"]].mean(axis=1)

    # --- 4. Soft Skills & Leadership ---
    soft_skill_cols = [c for c in [
        "communication_score", "teamwork_score", "leadership_score", "presentation_score"
    ] if c in df.columns]

    if soft_skill_cols:
        df["soft_skill_mean"] = df[soft_skill_cols].mean(axis=1)
        df["soft_skill_std"] = df[soft_skill_cols].std(axis=1).fillna(0)
        df["soft_skill_min"] = df[soft_skill_cols].min(axis=1)
        df["soft_skill_max"] = df[soft_skill_cols].max(axis=1)
        df["soft_skill_balance"] = df["soft_skill_max"] - df["soft_skill_min"]

    if all(c in df.columns for c in ["communication_score", "leadership_score"]):
        df["communication_leadership_gap"] = df["communication_score"] - df["leadership_score"]
        df['leadership_potential'] = (df['leadership_score'] + df['communication_score']) / 2

    if all(c in df.columns for c in ["teamwork_score", "presentation_score"]):
        df["teamwork_present_gap"] = df["teamwork_score"] - df["presentation_score"]

    # --- 5. Portfolio & Experience ---
    for c in [
        "real_client_project_count", "internship_count", "internship_duration_months",
        "freelance_project_count", "hackathon_count", "hackathon_awards",
        "github_repo_count", "github_avg_stars", "open_source_contribution_count",
        "certification_count", "bootcamp_count", "applications_sent", "interviews_attended"
    ]:
        if c in df.columns:
            df[f"log1p_{c}"] = np.log1p(pd.to_numeric(df[c], errors='coerce').fillna(0).clip(lower=0))

    exp_cols = [c for c in ["real_client_project_count", "freelance_project_count", "hackathon_count"] if c in df.columns]
    if exp_cols:
        df["total_projects"] = df[exp_cols].sum(axis=1)

    if all(c in df.columns for c in ["total_projects", "internship_duration_months"]):
        df["project_density"] = safe_div(df["total_projects"], 1 + df["internship_duration_months"])

    if all(c in df.columns for c in ["internship_count", "internship_duration_months"]):
        df['field_experience_index'] = (df['internship_count'] * 0.4) + (df['internship_duration_months'] * 0.6)

    if all(c in df.columns for c in ["hackathon_awards", "hackathon_count"]):
        df["hackathon_award_rate"] = safe_div(df["hackathon_awards"], 1 + df["hackathon_count"])

    if all(c in df.columns for c in ["github_avg_stars", "github_repo_count"]):
        df["repo_quality"] = df["github_avg_stars"].fillna(0) * np.log1p(df["github_repo_count"].fillna(0).clip(lower=0))
        df['github_impact_score'] = df['github_repo_count'] * df['github_avg_stars']

    if all(c in df.columns for c in ["open_source_contribution_count", "github_repo_count"]):
        df["open_source_intensity"] = safe_div(df["open_source_contribution_count"], 1 + df["github_repo_count"])

    # --- 6. Career Funnel & Professional Presence ---
    if all(c in df.columns for c in ["applications_sent", "interviews_attended"]):
        df["applications_per_interview"] = safe_div(df["applications_sent"], 1 + df["interviews_attended"])
        df["interview_conversion"] = safe_div(df["interviews_attended"], 1 + df["applications_sent"])
        df["application_pressure"] = df["applications_sent"] - df["interviews_attended"]

    profile_cols = [c for c in [
        "portfolio_score", "linkedin_profile_score", "cv_quality_score",
        "technical_interview_score", "hr_interview_score"
    ] if c in df.columns]

    if profile_cols:
        df["profile_signal"] = df[profile_cols].mean(axis=1)
        df["profile_signal_std"] = df[profile_cols].std(axis=1).fillna(0)

    if all(c in df.columns for c in ["linkedin_profile_score", "cv_quality_score"]):
        df['profile_quality'] = (df['linkedin_profile_score'] + df['cv_quality_score']) / 2

    if all(c in df.columns for c in ["technical_interview_score", "hr_interview_score"]):
        df['overall_interview_performance'] = (df['technical_interview_score'] + df['hr_interview_score']) / 2

    return df

# =========================================================
# 2. VERİLERİ OKU VE FEATURE ENGINEERING UYGULA
# =========================================================
train_data = pd.read_csv('train100.csv')
test_data = pd.read_csv('test100.csv')

# Önce matematiksel türetmeleri yapıyoruz! (Zayıf sütunların verilerini emiyoruz)
train_data = apply_feature_engineering(train_data)
test_data = apply_feature_engineering(test_data)

# =========================================================
# 3. ZAYIF ÖZELLİKLERİ (LOW IMPORTANCE) SİLME
# =========================================================
# Belirlediğin, modele katkısı olmayan gürültülü sütunların listesi:
cols_to_drop = [
    'devops_score', 'github_avg_stars', 'backend_score', 'linkedin_profile_score',
    'coding_score', 'hackathon_awards', 'applications_sent', 'certification_count',
    'leadership_score', 'english_exam_score', 'bootcamp_count', 'machine_learning_score',
    'sql_score', 'frontend_score', 'attendance_rate', 'hobby', 'hackathon_count',
    'data_structures_score', 'interviews_attended', 'department',
    'preferred_social_media_platform', 'internship_count', 'age', 'freelance_project_count'
]

# Sütunları hem Train hem Test setinden siliyoruz (errors='ignore' ile eğer zaten yoksa hata vermez)
train_data = train_data.drop(columns=cols_to_drop, errors='ignore')
test_data = test_data.drop(columns=cols_to_drop, errors='ignore')
print(f"Gereksiz {len(cols_to_drop)} sütun silindi. Kalan özellikler modele besleniyor...")

# =========================================================
# 4. TİPLERİ ZORLA (Casting)
# =========================================================
# (Silinen sütunlar listede olsa bile if col in df.columns kontrolü sayesinde hata vermez!)
categorical_cols = [
    'application_year', 'graduation_year', 'department', 'university_tier',
    'target_role', 'hobby', 'preferred_social_media_platform',
    'is_recent_grad', 'is_negative_grad_gap', 'failed_course_flag'
]

numeric_cols = [
    'age', 'cgpa', 'english_exam_score', 'attendance_rate', 'failed_courses_count', 'coding_score',
    'problem_solving_score', 'data_structures_score', 'sql_score', 'machine_learning_score',
    'backend_score', 'frontend_score', 'cloud_score', 'devops_score', 'project_quality_score',
    'real_client_project_count', 'internship_count', 'internship_duration_months', 'freelance_project_count',
    'hackathon_count', 'hackathon_awards', 'portfolio_score', 'github_repo_count', 'github_avg_stars',
    'open_source_contribution_count', 'linkedin_profile_score', 'cv_quality_score', 'technical_interview_score',
    'hr_interview_score', 'communication_score', 'teamwork_score', 'leadership_score', 'presentation_score',
    'certification_count', 'bootcamp_count', 'applications_sent', 'interviews_attended',

    # Yeni Üretilen Sayısallar
    'years_since_graduation', 'age_minus_years_since_grad', 'academic_mean', 'academic_std',
    'academic_min', 'academic_max', 'log_failed_courses_count', 'hard_skill_mean', 'hard_skill_std',
    'hard_skill_min', 'hard_skill_max', 'hard_skill_balance', 'frontend_backend_gap', 'ml_vs_web_gap',
    'ops_vs_app_gap', 'soft_skill_mean', 'soft_skill_std', 'soft_skill_min', 'soft_skill_max',
    'soft_skill_balance', 'communication_leadership_gap', 'teamwork_present_gap',
    'log1p_real_client_project_count', 'log1p_internship_count', 'log1p_internship_duration_months',
    'log1p_freelance_project_count', 'log1p_hackathon_count', 'log1p_hackathon_awards',
    'log1p_github_repo_count', 'log1p_github_avg_stars', 'log1p_open_source_contribution_count',
    'log1p_certification_count', 'log1p_bootcamp_count', 'log1p_applications_sent', 'log1p_interviews_attended',
    'total_projects', 'project_density', 'hackathon_award_rate', 'repo_quality', 'open_source_intensity',
    'applications_per_interview', 'interview_conversion', 'application_pressure', 'profile_signal',
    'profile_signal_std', 'academic_consistency_score', 'tech_versatility', 'core_tech_score',
    'leadership_potential', 'field_experience_index', 'github_impact_score', 'profile_quality',
    'overall_interview_performance'
]

text_cols = ['mentor_feedback_text']

for col in categorical_cols:
    if col in train_data.columns: train_data[col] = train_data[col].astype('category')
    if col in test_data.columns: test_data[col] = test_data[col].astype('category')

for col in numeric_cols:
    if col in train_data.columns: train_data[col] = pd.to_numeric(train_data[col], errors='coerce')
    if col in test_data.columns: test_data[col] = pd.to_numeric(test_data[col], errors='coerce')

for col in text_cols:
    if col in train_data.columns: train_data[col] = train_data[col].astype(str)
    if col in test_data.columns: test_data[col] = test_data[col].astype(str)

Gereksiz 24 sütun silindi. Kalan özellikler modele besleniyor...


In [2]:
# =========================================================
# 5. AUTOGLUON FORMATINA ÇEVİR VE EĞİTİME BAŞLA
# =========================================================
train_data = TabularDataset(train_data)
test_data = TabularDataset(test_data)

train_data = train_data.drop(columns=['student_id'], errors='ignore')
label = 'career_success_score'

print("Model eğitimi başlıyor. Lütfen bekleyin...")
predictor = TabularPredictor(
    label=label,
    eval_metric='root_mean_squared_error'
).fit(
    train_data=train_data,
    presets='best_quality',        # <--- Listede olan ve en yüksek skoru hedefleyen ana ayar
    hyperparameters='multimodal',  # <--- METNİ GPU VE DERİN ÖĞRENME İLE İŞLEMESİ İÇİN KRİTİK AYAR
    time_limit=1800,               # 2 Saat
    num_gpus=1
)

No path specified. Models will be saved in: "AutogluonModels/ag-20260614_154503"
Verbosity: 2 (Standard Logging)


Model eğitimi başlıyor. Lütfen bekleyin...


=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          12
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 39.49/39.49 GB
Total GPU Memory:   Free: 39.49 GB, Allocated: 0.00 GB, Total: 39.49 GB
GPU Count:          1
Memory Avail:       80.96 GB / 83.47 GB (97.0%)
Disk Space Avail:   54.52 GB / 112.64 GB (48.4%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='multimodal'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or di

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

INFO: Using 16bit Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO: You are using a CUDA device ('NVIDIA A100-SXM4-40GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/model_summary/model_summary.py:231: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.
INFO: 
  | Name              | Type                | Params | Mode 
------------------------------------------------------------------
0 | model             | MultimodalFusionMLP | 110 M  | train
1 | validat

In [ ]:
# =========================================================
# 6. TAHMİN VE SUBMISSION DOSYASI OLUŞTURMA
# =========================================================
predictions = predictor.predict(test_data)

submission_df = pd.DataFrame({
    'student_id': test_data['student_id'],
    'career_success_score': predictions
})

submission_df.to_csv('submission_autogluon_cleaned.csv', index=False)
print("İşlem tamamlandı! 'submission_autogluon_cleaned_multimodal.csv' dosyası oluşturuldu.")

In [ ]:
import pandas as pd
from autogluon.tabular import TabularDataset, TabularPredictor

# 1. Verileri Oku
train_data = pd.read_csv('train.csv')
test_data = pd.read_csv('test.csv')

# 2. Hangi sütun ne anlama geliyor tanımla
categorical_cols = ['application_year', 'graduation_year', 'department', 'university_tier', 'target_role', 'hobby', 'preferred_social_media_platform']
numeric_cols = [
    'age', 'cgpa', 'english_exam_score', 'attendance_rate', 'failed_courses_count', 'coding_score',
    'problem_solving_score', 'data_structures_score', 'sql_score', 'machine_learning_score',
    'backend_score', 'frontend_score', 'cloud_score', 'devops_score', 'project_quality_score',
    'real_client_project_count', 'internship_count', 'internship_duration_months', 'freelance_project_count',
    'hackathon_count', 'hackathon_awards', 'portfolio_score', 'github_repo_count', 'github_avg_stars',
    'open_source_contribution_count', 'linkedin_profile_score', 'cv_quality_score', 'technical_interview_score',
    'hr_interview_score', 'communication_score', 'teamwork_score', 'leadership_score', 'presentation_score',
    'certification_count', 'bootcamp_count', 'applications_sent', 'interviews_attended'
]
text_cols = ['mentor_feedback_text']

# 3. Tipleri Zorla
for col in categorical_cols:
    train_data[col] = train_data[col].astype('category')
    test_data[col] = test_data[col].astype('category')

for col in numeric_cols:
    train_data[col] = pd.to_numeric(train_data[col], errors='coerce')
    test_data[col] = pd.to_numeric(test_data[col], errors='coerce')

for col in text_cols:
    train_data[col] = train_data[col].astype(str)
    test_data[col] = test_data[col].astype(str)

# 4. AutoGluon Formatına Çevir ve Hedef Değişkeni Belirle
train_data = TabularDataset(train_data)
test_data = TabularDataset(test_data)

# Sadece eğitim verisinden ID'yi düşür (Testte lazım olacak)
train_data = train_data.drop(columns=['student_id'])
label = 'career_success_score'

In [ ]:
# Zaman sınırını saniye cinsinden belirliyoruz (Örn: 3600 = 1 Saat, 7200 = 2 Saat)
# Ne kadar uzun süre verirsen, o kadar çok model kombinasyonu dener.
time_limit_seconds = 3600

predictor = TabularPredictor(
    label=label,
    eval_metric='root_mean_squared_error' # Yarışmanın başarı kriterine göre bunu 'mse' veya 'r2' yapabilirsin
).fit(
    train_data=train_data,
    presets='best_quality',
    time_limit=time_limit_seconds,
    num_gpus=1  # Colab'daki GPU'yu kullanmasını zorluyoruz
)

No path specified. Models will be saved in: "AutogluonModels/ag-20260613_223855"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          12
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 39.49/39.49 GB
Total GPU Memory:   Free: 39.49 GB, Allocated: 0.00 GB, Total: 39.49 GB
GPU Count:          1
Memory Avail:       81.04 GB / 83.47 GB (97.1%)
Disk Space Avail:   58.34 GB / 112.64 GB (51.8%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=Tr

In [ ]:
# Liderlik tablosu: Hangi algoritmaların tek tek ne kadar iyi performans gösterdiğini listeler
predictor.leaderboard(train_data)

# Özellik Önemi (Feature Importance): Hangi sütunların tahmine ne kadar katkı sağladığını hesaplar
feature_importance = predictor.feature_importance(train_data)
display(feature_importance)

Computing feature importance via permutation shuffling for 45 features using 5000 rows with 5 shuffle sets...
	3929.44s	= Expected runtime (785.89s per shuffle set)


In [ ]:
# İçine train_data KOYMUYORUZ!
leaderboard_df = predictor.leaderboard()
display(leaderboard_df)

,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L3,-8.749655,root_mean_squared_error,43.702515,2168.439713,0.000559,0.036385,3,True,21
1,WeightedEnsemble_L2,-8.768181,root_mean_squared_error,3.721605,731.953965,0.000628,0.028696,2,True,13
2,CatBoost_BAG_L2,-8.802544,root_mean_squared_error,30.767509,1792.332730,1.000825,64.094888,2,True,17
3,LightGBMXT_BAG_L2,-8.834181,root_mean_squared_error,29.987591,1761.565217,0.220907,33.327375,2,True,14
4,LightGBM_BAG_L2,-8.843385,root_mean_squared_error,29.980403,1762.058212,0.213719,33.820370,2,True,15
5,RandomForestMSE_BAG_L2,-8.844279,root_mean_squared_error,42.095678,1998.881020,12.328994,270.643178,2,True,16
6,XGBoost_BAG_L2,-8.856228,root_mean_squared_error,30.522746,1790.314930,0.756063,62.077088,2,True,20
7,ExtraTreesMSE_BAG_L2,-8.865842,root_mean_squared_error,42.202458,2034.762715,12.435775,306.524873,2,True,18
8,CatBoost_r177_BAG_L1,-8.947408,root_mean_squared_error,0.967304,116.505059,0.967304,116.505059,1,True,10
9,LightGBM_r131_BAG_L1,-8.969899,root_mean_squared_error,0.593387,64.993557,0.593387,64.993557,1,True,12


In [ ]:
# Test verisi üzerinden tahmin yap
predictions = predictor.predict(test_data)

# Kaggle formatında bir DataFrame oluştur
submission_df = pd.DataFrame({
    'student_id': test_data['student_id'],
    'career_success_score': predictions
})

# CSV olarak dışa aktar (Colab sol panelindeki klasör ikonunda belirecek, oradan indirebilirsin)
submission_df.to_csv('submission_autogluon_a100_ilk.csv', index=False)
print("Submission dosyası başarıyla oluşturuldu!")

Submission dosyası başarıyla oluşturuldu!


In [ ]:
# Sıkıştırma işlemi
!zip -r autogluon_ilk_özelliksiz_hamveri_a100.zip /content/AutogluonModels

# İndirme işlemi
from google.colab import files
files.download('autogluon_ilk_özelliksiz_hamveri_a100.zip')

## Conclusion

In this quickstart tutorial we saw AutoGluon's basic fit and predict functionality using `TabularDataset` and `TabularPredictor`. AutoGluon simplifies the model training process by not requiring feature engineering or model hyperparameter tuning. Next, we recommend checking out the [Essentials Tutorial](tabular-essentials.ipynb) to learn about `presets` to use for production and competition usage. You can also check out the [in-depth tutorials](index.html) to learn more about AutoGluon's other features like customizing the training and prediction steps or extending AutoGluon with custom feature generators, models, or metrics.